# Study: Contact Surface Green's Function

This notebook studies the retarded surface Green's function $g^R$ of a semi-infinite contact and links a toy model to the open-boundary recursion used in `quatrex`.

## 1) Scalar contact model

For a nearest-neighbor semi-infinite chain with onsite energy $\varepsilon_0$ and hopping $t$,

$$g^R = [z - t^2 g^R]^{-1}, \quad z = E + i\eta - \varepsilon_0.$$

This gives

$$t^2(g^R)^2 - z g^R + 1 = 0.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def g_surface_analytic(E, eps0=0.0, t=1.0, eta=1e-6):
    z = E + 1j * eta - eps0
    disc = np.sqrt(z**2 - 4 * t**2)
    g1 = (z + disc) / (2 * t**2)
    g2 = (z - disc) / (2 * t**2)
    return np.where(np.imag(g1) <= 0, g1, g2)


In [ ]:
eps0 = 0.0
t = 1.0
eta = 1e-6
E = np.linspace(-3.0, 3.0, 1200)

g = g_surface_analytic(E, eps0=eps0, t=t, eta=eta)
z = E + 1j * eta - eps0
quad_residual = t**2 * g**2 - z * g + 1
print(f"Max quadratic residual: {np.max(np.abs(quad_residual)):.3e}")
print(f"Max Im(g^R): {np.max(np.imag(g)):.3e} (retarded branch should be <= 0)")


In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
ax[0].plot(E, np.real(g))
ax[0].set_ylabel("Re(g^R)")

ax[1].plot(E, np.imag(g))
ax[1].set_ylabel("Im(g^R)")

dos = -np.imag(g) / np.pi
ax[2].plot(E, dos)
ax[2].set_ylabel("surface DOS")
ax[2].set_xlabel("Energy E")

for a in ax:
    a.axvline(-2 * t + eps0, color="k", lw=0.8, alpha=0.4)
    a.axvline(2 * t + eps0, color="k", lw=0.8, alpha=0.4)
    a.grid(True, alpha=0.25)

fig.suptitle("Surface Green's function and DOS for a semi-infinite 1D contact")
plt.tight_layout()
plt.show()


## 2) Matrix recursion used in `quatrex`

`quatrex` solves

$$x_{ii} = (a_{ii} - a_{ji} x_{ii} a_{ij})^{-1}.$$

The cell below applies this to a simple 2x2 contact block and checks convergence.

In [ ]:
def matrix_fixed_point(a_ji, a_ii, a_ij, niter=1200):
    x = np.linalg.inv(a_ii)
    for _ in range(niter):
        x = np.linalg.inv(a_ii - a_ji @ x @ a_ij)
    return x

E0 = 0.3
eta0 = 1e-2
eps = np.array([[0.0, 0.05], [0.05, 0.2]])
coupling = np.array([[-0.3, 0.0], [0.0, -0.24]])

a_ii = (E0 + 1j * eta0) * np.eye(2) - eps
a_ji = coupling
a_ij = coupling.T

x = matrix_fixed_point(a_ji, a_ii, a_ij)
residual = np.linalg.norm(x - np.linalg.inv(a_ii - a_ji @ x @ a_ij))
print("Surface Green function estimate x_ii:")
print(x)
print(f"Fixed-point residual norm: {residual:.3e}")


## 3) Optional comparison with `qttools` Sancho-Rubio

If `qttools` is importable from the notebook environment, compare with `SanchoRubio`.

In [ ]:
try:
    from qttools.boundary_conditions.obc.sancho_rubio import SanchoRubio

    sancho = SanchoRubio(max_iterations=400, convergence_tol=1e-12)
    x_sr = sancho((a_ji, a_ii, a_ij), contact="left")
    print("||x_sr - x_fixed_point|| =", np.linalg.norm(x_sr - x))
except Exception as exc:
    print("SanchoRubio comparison skipped:", exc)
